# 01c 데이터 수집: 구글 별점·가격대·식당 특징 (공식 Places API)

노트북 01에서 모은 카카오 식당 목록에 구글 지도의 **별점, 가격대, 점심·저녁 영업, 단체석·예약, 영업시간, 최근 리뷰 글**을 붙입니다. 구글이 공식으로 제공하는 Places API를 쓰므로 약관 문제가 없습니다.

| 항목 | 내용 |
|---|---|
| 입력 | `restaurants.csv` (노트북 01 결과) |
| 출력 | `google_places.csv` (카카오 식당 1곳 = 1행), `google_reviews.csv` (리뷰 1건 = 1행, 식당당 최대 5건) |
| 받지 않는 것 | 리뷰 작성자 이름·사진 등 개인정보 |
| 방법 | 식당마다 "이름 + 회사 근처 좌표"로 구글에서 검색 → 가장 비슷한 곳을 골라 별점을 붙임 |
| 키 | Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `GOOGLE_PLACES_API_KEY` |

### 받는 정보와 쓰는 곳
| 구글 필드 | 뜻 | 우리 설계에서 쓰는 곳 |
|---|---|---|
| rating, userRatingCount | 별점, 별점 수 | 인기도 추천, 보정평균 |
| priceLevel | 가격 수준 4단계 (저렴~아주 비쌈) | 예산 조건 필터 |
| priceRange | 1인 가격 범위 (예: 1만~2만 원) | 예산 조건 필터 |
| servesLunch, servesDinner | 점심·저녁 영업 여부 | 카테고리 1단계 (점심/저녁) |
| goodForGroups, reservable | 단체 적합, 예약 가능 | 목적: 팀 회식 |
| regularOpeningHours | 영업시간 | 점심·저녁 영업 확인 |
| reviews (최대 5건, 글만) | 최근 리뷰 글 | 한줄평 키워드 분석 (21~23강) |

**주의:** 가격·점심 여부·단체석 같은 정보는 **구글에 등록된 식당만** 있습니다. 한국 식당은 빈칸이 많을 수 있어서, 5곳 시험 때 얼마나 채워지는지 꼭 확인합니다.

### 💰 비용 (먼저 읽어 주세요)
- 식당 1곳 = 검색 **1번**. 식사 식당 1,243곳이면 **1,243번**입니다.
- 가격·단체석·리뷰 글을 함께 받으면 구글 요금표에서 가장 높은 **"Text Search Enterprise + Atmosphere"** 등급으로 계산됩니다. 별점만 받을 때보다 비싸지만 **한 달 무료 사용량은 똑같이 약 1,000번**입니다. 넘으면 1,000번당 약 40달러입니다. *(2025년 기준 요금표로 추정한 값입니다. 실행 전 [구글 요금표](https://developers.google.com/maps/billing-and-pricing/pricing)에서 확인하세요.)*
- 그래서 기본 설정은 **이번 달 950번까지만** 검색하고 멈춥니다 → **무료**.
- 나머지 약 290곳은 ① 다음 달(11월 1일 이후) 다시 실행하거나 ② `MAX_CALLS`를 늘려 지금 받으면 약 **12달러(약 1만 7천 원)** 정도 나옵니다.
- 이미 받은 식당은 다시 검색하지 않으므로, 다시 실행해도 돈이 두 번 나가지 않습니다.
- 안전장치: 구글 클라우드 콘솔에서 이 키의 **하루 사용량 한도**를 1,000으로 걸어 두면 실수로 많이 써도 막힙니다.

In [ ]:
# ── 설정 ─────────────────────────────────────────
MAX_CALLS = 950            # 이번 실행에서 최대 검색 횟수 (무료 범위)
TEST_ONLY = True           # True: 5곳만 시험. 결과가 괜찮으면 False로 바꾸고 다시 실행
NOT_MEAL = ["간식", "식품판매", "구내식당", "푸드코트"]
MATCH_RADIUS_M = 150       # 카카오 위치에서 이 거리 안의 구글 장소만 같은 식당 후보로 봄

## 1. 준비: 키와 식당 목록

In [ ]:
import os, time, math, re, difflib
import pandas as pd
import requests

def load_key():
    try:
        from google.colab import userdata
        return userdata.get("GOOGLE_PLACES_API_KEY")
    except ImportError:
        pass
    key = os.environ.get("GOOGLE_PLACES_API_KEY")
    if not key:
        from getpass import getpass
        key = getpass("구글 Places API 키: ")
    return key

API_KEY = load_key()

path = "restaurants.csv"
if not os.path.exists(path):
    try:
        from google.colab import files
        print("restaurants.csv 를 골라 주세요")
        path = list(files.upload().keys())[0]
    except ImportError:
        path = "../data/raw/restaurants.csv"
rest = pd.read_csv(path)
meal = rest[~rest["menu_category"].isin(NOT_MEAL)].sort_values("distance_m").reset_index(drop=True)
print("식사 식당:", len(meal), "곳 (회사에서 가까운 순으로 검색)")

# 이어 받기: 예전에 받은 결과가 있으면 그 식당은 건너뜀
OUT = "google_places.csv"
OUT_REVIEWS = "google_reviews.csv"
done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()
done_ids = set(done["restaurant_id"]) if len(done) else set()
todo = meal[~meal["restaurant_id"].isin(done_ids)]
print("이미 받은 곳:", len(done_ids), "/ 남은 곳:", len(todo))

## 2. 구글 검색 함수와 "같은 식당인지" 판단 규칙

카카오와 구글은 이름 표기가 조금씩 다릅니다 (예: `차이홍` vs `차이홍 여의도`). 그래서 두 가지를 같이 봅니다.
- **거리:** 카카오 위치에서 150m 안에 있는가
- **이름 닮은 정도:** 띄어쓰기·지점명을 빼고 비교한 점수 (0~1)

| 판정 | 기준 |
|---|---|
| 일치 | 150m 안 + 이름 점수 0.6 이상 (또는 한쪽 이름이 다른 쪽에 포함) |
| 애매 | 150m 안 + 이름 점수 0.4~0.6 → 별점은 붙이되 표시해 둠 |
| 실패 | 그 밖 → 별점 빈칸 |

In [ ]:
URL = "https://places.googleapis.com/v1/places:searchText"
FIELDS = ",".join("places." + f for f in [
    "id", "displayName", "location", "rating", "userRatingCount",
    "priceLevel", "priceRange", "servesLunch", "servesDinner",
    "goodForGroups", "reservable", "regularOpeningHours", "reviews",
])
PRICE_LEVEL_KO = {"PRICE_LEVEL_FREE": "무료", "PRICE_LEVEL_INEXPENSIVE": "저렴",
                  "PRICE_LEVEL_MODERATE": "보통", "PRICE_LEVEL_EXPENSIVE": "비쌈",
                  "PRICE_LEVEL_VERY_EXPENSIVE": "아주 비쌈"}

def won(price):
    # {"currencyCode": "KRW", "units": "10000"} → 10000
    try:
        return int(price["units"]) if price.get("currencyCode", "KRW") == "KRW" else None
    except (KeyError, TypeError, ValueError):
        return None

def search_google(name, lat, lng):
    body = {
        "textQuery": name,
        "languageCode": "ko",
        "regionCode": "KR",
        "maxResultCount": 5,
        "locationBias": {"circle": {"center": {"latitude": lat, "longitude": lng}, "radius": 300.0}},
    }
    r = requests.post(URL, json=body, timeout=15,
                      headers={"X-Goog-Api-Key": API_KEY, "X-Goog-FieldMask": FIELDS})
    if r.status_code == 403:
        raise RuntimeError("권한 오류: 구글 클라우드에서 'Places API (New)'를 사용 설정했는지, 결제 계정이 연결됐는지 확인하세요.\n" + r.text[:300])
    if r.status_code == 429:
        raise RuntimeError("사용량 한도에 걸렸습니다. 여기서 멈춥니다.")
    r.raise_for_status()
    return r.json().get("places", [])

def distance_m(lat1, lng1, lat2, lng2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lng2 - lng1) / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def norm(s):
    s = re.sub(r"\(.*?\)", "", str(s))
    s = re.sub(r"\s+\S*(점|지점|본점)$", "", s.strip())   # 띄어 쓴 끝 단어의 지점명 제거 (예: "샐러디 여의도역점")
    s = s.replace("여의도", "")
    return re.sub(r"[^0-9a-zA-Z가-힣]", "", s).lower()

def name_score(a, b):
    na, nb = norm(a), norm(b)
    if not na or not nb:
        return 0.0
    if na in nb or nb in na:
        return 1.0
    return difflib.SequenceMatcher(None, na, nb).ratio()

def best_match(row, places):
    best = None
    for p in places:
        loc = p.get("location", {})
        if "latitude" not in loc:
            continue
        d = distance_m(row["y"], row["x"], loc["latitude"], loc["longitude"])
        if d > MATCH_RADIUS_M:
            continue
        s = name_score(row["name"], p.get("displayName", {}).get("text", ""))
        if best is None or s > best[0] or (s == best[0] and d < best[1]):
            best = (s, d, p)
    if best is None or best[0] < 0.4:
        return "실패", None, None, None
    return ("일치" if best[0] >= 0.6 else "애매"), best[0], best[1], best[2]

## 3. 검색하기

In [ ]:
limit = 5 if TEST_ONLY else MAX_CALLS
old_reviews = pd.read_csv(OUT_REVIEWS) if os.path.exists(OUT_REVIEWS) else pd.DataFrame()

def save_reviews(review_rows):
    pd.concat([old_reviews, pd.DataFrame(review_rows)], ignore_index=True).to_csv(OUT_REVIEWS, index=False, encoding="utf-8-sig")

rows, review_rows, calls = [], [], 0
for _, row in todo.head(limit).iterrows():
    try:
        places = search_google(row["name"], row["y"], row["x"])
    except RuntimeError as e:
        print(e); break
    calls += 1
    quality, score, dist, p = best_match(row, places)
    p = p or {}
    pr = p.get("priceRange", {})
    hours = p.get("regularOpeningHours", {})
    rows.append({
        "restaurant_id": row["restaurant_id"],
        "google_name": p.get("displayName", {}).get("text"),
        "google_place_id": p.get("id"),
        "google_rating": p.get("rating"),
        "google_rating_count": p.get("userRatingCount", 0) if p else None,
        "price_level": PRICE_LEVEL_KO.get(p.get("priceLevel")),
        "price_min_won": won(pr.get("startPrice", {})),
        "price_max_won": won(pr.get("endPrice", {})),
        "serves_lunch": p.get("servesLunch"),
        "serves_dinner": p.get("servesDinner"),
        "good_for_groups": p.get("goodForGroups"),
        "reservable": p.get("reservable"),
        "opening_hours": " / ".join(hours.get("weekdayDescriptions", [])) or None,
        "match_quality": quality,
        "name_score": round(score, 2) if score is not None else None,
        "match_distance_m": round(dist) if dist is not None else None,
        "collected_at": pd.Timestamp.today().date().isoformat(),
    })
    # 리뷰는 글·별점·작성 시기만 저장 (작성자 이름·사진은 저장하지 않음)
    if quality != "실패":
        for rv in p.get("reviews", [])[:5]:
            review_rows.append({
                "restaurant_id": row["restaurant_id"],
                "review_rating": rv.get("rating"),
                "review_text": (rv.get("text") or rv.get("originalText") or {}).get("text"),
                "review_language": (rv.get("text") or rv.get("originalText") or {}).get("languageCode"),
                "review_time": rv.get("relativePublishTimeDescription"),
                "source": "구글 Places API",
            })
    if calls % 50 == 0:
        print(f"{calls}번 검색 완료")
        pd.concat([done, pd.DataFrame(rows)]).to_csv(OUT, index=False, encoding="utf-8-sig")  # 중간 저장
        save_reviews(review_rows)
    time.sleep(0.2)

new = pd.DataFrame(rows)
save_reviews(review_rows)
result = pd.concat([done, new], ignore_index=True)
result.to_csv(OUT, index=False, encoding="utf-8-sig")
print(f"이번 검색 {calls}번 / 누적 {len(result)}곳 저장")
new.merge(meal[["restaurant_id", "name"]], on="restaurant_id")[["name", "google_name", "google_rating", "price_level", "price_min_won", "serves_lunch", "good_for_groups", "match_quality"]].head(10)

## 4. 검증: 매칭이 잘 됐나

`TEST_ONLY = True`일 때는 여기서 5곳 결과를 보고, 이름이 같은 식당끼리 짝지어졌으면 설정을 `False`로 바꿔 다시 **모두 실행**하세요.

In [ ]:
chk = result.merge(meal[["restaurant_id", "name", "menu_category"]], on="restaurant_id")
print(chk["match_quality"].value_counts().to_string())
print("별점 있는 곳:", chk["google_rating"].notna().sum(), "/", len(chk))
print("별점 범위:", chk["google_rating"].min(), "~", chk["google_rating"].max(), "(1~5 안이어야 정상)")
print()
print("항목별로 채워진 비율 (매칭 성공한 곳 기준, %)")
ok = chk[chk["match_quality"] != "실패"]
fill_cols = ["google_rating", "price_level", "price_min_won", "serves_lunch", "serves_dinner", "good_for_groups", "reservable", "opening_hours"]
print(ok[fill_cols].notna().mean().mul(100).round(0).to_string())
reviews = pd.read_csv(OUT_REVIEWS) if os.path.exists(OUT_REVIEWS) else pd.DataFrame()
print("리뷰 글:", len(reviews), "건")
print()
print("'애매' 판정 예시 (눈으로 확인)")
chk[chk["match_quality"] == "애매"][["name", "google_name", "name_score", "match_distance_m"]].head(10)

In [ ]:
# 메뉴별로 매칭 성공률이 크게 다르면 한쪽으로 치우친 데이터가 됩니다 (보고서에 적을 내용)
chk.assign(성공=chk["match_quality"] != "실패").groupby("menu_category")["성공"].mean().mul(100).round(0).sort_values().head(10)

## 5. 내려받기 (깃허브에는 올리지 않습니다. 리뷰 글은 구글 콘텐츠라 원본 공개 금지)

In [ ]:
try:
    from google.colab import files
    files.download(OUT)
    files.download(OUT_REVIEWS)
except ImportError:
    pass